# 02 - M2 texto (TF-IDF 5k + XGBoost): entrenamiento honesto + graficas

Cuaderno de codigo y graficas (markdown minimo). Entrena M2 con ventanas UTC fijas, guarda modelo y metricas, y deja confusion, PR, tokens, scores, comparacion M1/M2, latencia y errores.



In [ ]:
import json, pickle, sys, time
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.sparse import hstack
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
    f1_score, average_precision_score, precision_recall_curve, roc_curve)
from xgboost import XGBClassifier
ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.temporal_split import WINDOW_LABELS, population_summary, temporal_windows
CATS = ["message_type","stream_name","product_name","vendor_name","pipeline","org_id","action","severity","protocol"]
NUMS = ["hour","msg_len","lag_h"]
TEXT = "message_sanitized"
FIG = ROOT / "reports" / "figures"
print("ROOT:", ROOT)
print("m1 committed:", json.loads((ROOT/"models"/"m1_metrics.json").read_text())["f1_macro"])



In [ ]:
def features(df):
    df = df.copy()
    ts = pd.to_datetime(df["timestamp"], unit="s", utc=True)
    df["hour"] = ts.dt.hour
    df["msg_len"] = df[TEXT].fillna("").str.len()
    df["lag_h"] = ((df["timestamp"]-df["event_time"])/3600).clip(-1e4,1e4).fillna(-1)
    for c in CATS:
        df[c] = df[c].fillna("").astype(str)
    df[TEXT] = df[TEXT].fillna("")
    return df
def sample30k(df, n):
    mal = df[df.label_binary=="malicious"]
    rest = df[df.label_binary!="malicious"]
    n_rest = max(n-len(mal), 2000)
    sus = rest[rest.label_binary=="suspicious"]; ben = rest[rest.label_binary=="benign"]
    n_sus = min(len(sus), n_rest//3); n_ben = min(len(ben), n_rest-n_sus)
    out = pd.concat([mal, sus.sample(n_sus, random_state=42), ben.sample(n_ben, random_state=42)])
    return out.sample(frac=1.0, random_state=42).reset_index(drop=True)
cols = ["label_binary","timestamp","event_time",TEXT]+CATS
raw = pd.read_parquet(ROOT/"data"/"signals"/"signals.parquet", columns=cols)
train_pop, test, drift = temporal_windows(raw)
print("ventanas:", WINDOW_LABELS)
print("test prevalencia real:", population_summary(test))
train_df = features(sample30k(train_pop, 30000)); test_df = features(test)
X_train, y_train = train_df[CATS+NUMS+[TEXT]], train_df["label_binary"]
X_test, y_test = test_df[CATS+NUMS+[TEXT]], test_df["label_binary"]
print("train/test:", len(y_train), len(y_test), y_train.value_counts().to_dict())



In [ ]:
vec = TfidfVectorizer(max_features=5000, ngram_range=(1,2), sublinear_tf=True, min_df=5)
Ttr = vec.fit_transform(X_train[TEXT].tolist()); Tte = vec.transform(X_test[TEXT].tolist())
tab = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore", max_categories=30), CATS), ("num","passthrough",NUMS)])
Atr = tab.fit_transform(X_train[CATS+NUMS]); Ate = tab.transform(X_test[CATS+NUMS])
Xtr = hstack([Ttr,Atr]).tocsr(); Xte = hstack([Tte,Ate]).tocsr()
print("Xtr/Xte:", Xtr.shape, Xte.shape, "| vocab:", len(vec.vocabulary_))



In [ ]:
def recall_at_fpr(y_true, scores, fpr_max=0.01):
    fpr, tpr, _ = roc_curve(y_true, scores)
    ok = fpr <= fpr_max
    return round(float(tpr[ok].max()) if ok.any() else 0.0, 4)
le = LabelEncoder().fit(sorted(y_train.unique().tolist()))
ytr, yte = le.transform(y_train), le.transform(y_test)
counts = y_train.value_counts(); sw = y_train.map(len(y_train)/(len(counts)*counts)).to_numpy()
t0 = time.time()
clf = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.08, subsample=0.8,
    colsample_bytree=0.8, tree_method="hist", objective="multi:softprob", num_class=3,
    eval_metric="mlogloss", n_jobs=-1, random_state=42)
clf.fit(Xtr, ytr, sample_weight=sw)
print("train_s:", round(time.time()-t0,1))
p = le.inverse_transform(clf.predict(Xte)); proba = clf.predict_proba(Xte)
classes = le.classes_.tolist()
rep = classification_report(y_test, p, output_dict=True, zero_division=0)
f1m = f1_score(y_test, p, average="macro")
print(classification_report(y_test, p, zero_division=0)); print("F1-macro:", round(f1m,4))
per_class = {}
for i,c in enumerate(classes):
    auc = round(float(average_precision_score((yte==i).astype(int), proba[:,i])),4)
    per_class[c] = {"precision":round(rep[c]["precision"],4),"recall":round(rep[c]["recall"],4),
        "f1":round(rep[c]["f1-score"],4),"support":int(rep[c]["support"]),"pr_auc":auc}
mal_idx = classes.index("malicious")
r1 = recall_at_fpr((yte==mal_idx).astype(int), proba[:,mal_idx])
print("recall mal @1%FPR:", r1)
metrics = {"model":"m2-texto-v1","encoder":"tfidf-5k","f1_macro":round(float(f1m),4),
    "recall_mal_at_1fpr":r1,"per_class":per_class,"n_train":len(y_train),"n_test":len(y_test),
    "split":"fixed UTC windows, 0 incidentes compartidos","sample_n":30000}
(ROOT/"models"/"m2_metrics.json").write_text(json.dumps(metrics, indent=2))
with open(ROOT/"models"/"m2.pkl","wb") as f:
    pickle.dump({"vectorizer":vec,"encoder":"tfidf-5k","tab":tab,"clf":clf,"le":le,
                 "meta":{"cats":CATS,"nums":NUMS}}, f)
print("guardado models/m2.pkl + m2_metrics.json")



In [ ]:
fig, ax = plt.subplots(figsize=(7,6))
ConfusionMatrixDisplay(confusion_matrix(y_test, p, labels=classes), display_labels=classes).plot(ax=ax, colorbar=False)
ax.set_title("M2: matriz de confusion (test futuro, prevalencia real)")
fig.tight_layout(); fig.savefig(FIG/"confusion_m2.pdf"); plt.show()
print("guardado reports/figures/confusion_m2.pdf")



In [ ]:
fig, ax = plt.subplots(figsize=(7,5))
for i,c in enumerate(classes):
    prec, rec, _ = precision_recall_curve((yte==i).astype(int), proba[:,i])
    ax.plot(rec, prec, label=c+" (AUC %.3f)" % per_class[c]["pr_auc"])
ax.set_xlabel("recall"); ax.set_ylabel("precision"); ax.set_title("M2: curvas PR por clase")
ax.legend(); fig.tight_layout(); fig.savefig(FIG/"pr_m2.pdf"); plt.show()
print("guardado reports/figures/pr_m2.pdf")



In [ ]:
terms = np.array(vec.get_feature_names_out())
Mm = Tte[yte==le.transform(["malicious"])[0]].mean(axis=0).A1
Mb = Tte[yte==le.transform(["benign"])[0]].mean(axis=0).A1
gap = Mm - Mb
top = np.argsort(gap)[-20:][::-1]
fig, ax = plt.subplots(figsize=(8,7))
ax.barh(terms[top][::-1], gap[top][::-1])
ax.set_title("Top-20 terminos TF-IDF: media malicious menos media benign")
fig.tight_layout(); fig.savefig(FIG/"tokens_m2.pdf"); plt.show()
print("guardado reports/figures/tokens_m2.pdf")
print("top-5:", terms[top[:5]].tolist())



In [ ]:
fig, axes = plt.subplots(1,2,figsize=(12,4))
axes[0].hist(proba[yte==mal_idx, mal_idx], bins=40, alpha=0.7, label="malicious real")
axes[0].hist(proba[yte!=mal_idx, mal_idx], bins=40, alpha=0.7, label="resto")
axes[0].set_title("M2: P(malicious) por tier real"); axes[0].legend()
m1 = json.loads((ROOT/"models"/"m1_metrics.json").read_text())["per_class"]
labels = ["F1-macro","recall mal","PR-AUC mal"]
m1v = [json.loads((ROOT/"models"/"m1_metrics.json").read_text())["f1_macro"],
       m1["malicious"]["recall"], m1["malicious"]["pr_auc"]]
m2v = [metrics["f1_macro"], per_class["malicious"]["recall"], per_class["malicious"]["pr_auc"]]
x = np.arange(len(labels)); w = 0.35
axes[1].bar(x-w/2, m1v, w, label="M1 tabular (split cuantil, committed)")
axes[1].bar(x+w/2, m2v, w, label="M2 texto (ventanas fijas, este cuaderno)")
axes[1].set_xticks(x); axes[1].set_xticklabels(labels); axes[1].legend(); axes[1].set_title("M1 vs M2")
fig.tight_layout(); fig.savefig(FIG/"compare_m1_m2.pdf"); plt.show()
print("guardado reports/figures/compare_m1_m2.pdf")
print("M1 committed vs M2 honesto:", [round(a,4) for a in m1v], "vs", [round(b,4) for b in m2v])



In [ ]:
for bs in [1, 32, 500]:
    s = Xte[:bs]
    t0=time.time(); n=20
    for _ in range(n):
        _ = clf.predict_proba(s)
    ms = (time.time()-t0)/n/bs*1000
    print("batch %d: %.2f ms/evento (solo inferencia XGB)" % (bs, ms))
t0=time.time(); _ = vec.transform(X_test[TEXT].tolist()); print("vectorizar test completo: %.1fs" % (time.time()-t0))



In [ ]:
pmal = proba[:,mal_idx]
fn = np.where((y_test.to_numpy()=="malicious") & (le.inverse_transform(clf.predict(Xte))!="malicious"))[0]
print("FN malicious:", len(fn), "de", int((y_test=="malicious").sum()))
for j in fn[:3]:
    print("--- FN score=%.3f msg=%s" % (pmal[j], X_test[TEXT].iloc[j][:300].replace(chr(10)," ")))
fp = np.argsort(pmal[yte!=mal_idx])[::-1][:3]
ni = np.where(yte!=mal_idx)[0][fp]
print("top-3 FP benignos:")
for j in ni:
    print("--- FP score=%.3f tier=%s msg=%s" % (pmal[j], y_test.iloc[j], X_test[TEXT].iloc[j][:300].replace(chr(10)," ")))

